# Backtesting: exceptions and independence tests

Backtests every VaR in the saved GARCH results over its forecast window, with the Kupiec
(unconditional coverage) and Christoffersen (independence) tests.

When prices end inside a forecast window, the detail is padded with calendar dates through
the window end automatically, so there are no dates to update by hand. Pass
`pad_incomplete=False` to turn this off.

Settings come from `configs/var_config_volfactor.yml`, found automatically in this folder or
any parent folder. To use a different config or price file (same format, different data):

```python
vf.set_config("path/to/config.yml")
vf.set_data("path/to/prices.csv")   # or a .parquet file, or a DataFrame
```

In [ ]:
import pandas as pd
import volatility_factor as vf

cfg = vf.get_config()
results = pd.read_excel(f"{cfg.output_file}.xlsx", dtype={"Weights": str})
results.head()

**Run all backtests** and save `backtesting_detail_<name>.parquet` and `independence_tests_<name>.xlsx`

In [ ]:
bt_run = vf.run_backtests(results, converge_rate=0.01, alpha=0.99).save()
bt_run.independence.head()

In [ ]:
bt_run.detail.head()

**One backtest in detail**

In [ ]:
row = results.iloc[0]
bt = vf.VarData.from_id(row.Volatility_Factor_ID).backtest(
    n_liquidation=int(row.NDays), var=row.VaR_99,
    start_date=pd.to_datetime(row.ForecastStart).strftime("%Y-%m-%d"),
    end_date=pd.to_datetime(row.ForecastEnd).strftime("%Y-%m-%d"),
    var_is_log=False, forward_shift=True,
)
bt.independence_test()

In [ ]:
bt.backtest_plot()